# Notebook 6 — Architecture des LLMs
### BNP Paribas — Senior AI Engineer

---
## Sommaire
1. [Vue d'ensemble : Transformer original](#transformer)
2. [Self-Attention en profondeur](#selfattention)
3. [Encoder-Only (BERT, RoBERTa)](#encoder)
4. [Decoder-Only (GPT, LLaMA)](#decoder)
5. [Encoder-Decoder (T5, BART)](#encdec)
6. [Positional Encodings](#pe)
7. [Optimisations architecturales modernes](#optim)
8. [Comparaison des architectures](#compare)
9. [Questions d’entretien](#questions)


---
## 1. Vue d’ensemble : Transformer original (Vaswani et al., 2017) <a name='transformer'></a>

### Architecture
```
Input Tokens
     |
[Token Embedding + Positional Encoding]
     |
+------------------------------------+
|         ENCODER BLOCK x N         |
|  [Multi-Head Self-Attention]       |
|  [Add & Norm (LayerNorm)]          |
|  [Feed-Forward Network]            |
|  [Add & Norm]                      |
+------------------------------------+
     |  (encoder output = memory)
+------------------------------------+
|         DECODER BLOCK x N         |
|  [Masked Multi-Head Self-Attention]|
|  [Add & Norm]                      |
|  [Cross-Attention (Q from dec,     |
|   K/V from encoder)]               |
|  [Add & Norm]                      |
|  [Feed-Forward Network]            |
|  [Add & Norm]                      |
+------------------------------------+
     |
[Linear + Softmax]
     |
Output probabilities
```

### Hyper-parametres du Transformer original (base)
| Param | Valeur | Description |
|---|---|---|
| d_model | 512 | Dimension des embeddings |
| n_heads | 8 | Nombre de tetes d attention |
| d_k = d_v | 64 | Dimension par tete (d_model/n_heads) |
| d_ff | 2048 | Dimension du FFN (4 * d_model) |
| N | 6 | Nombre de blocs encoder/decoder |
| dropout | 0.1 | Regularisation |


In [ ]:
import numpy as np

# ============================================================
# TRANSFORMER BLOCK COMPLET -- from scratch
# ============================================================

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=-1, keepdims=True) + 1e-10)

def layer_norm(x, gamma, beta, eps=1e-6):
    mu  = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return gamma * (x - mu) / np.sqrt(var + eps) + beta

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

class TransformerBlock:
    """
    Un bloc Transformer complet:
      x -> MHA -> Add&Norm -> FFN -> Add&Norm
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads
        self.d_ff    = d_ff
        self.dropout = dropout

        scale = np.sqrt(2.0 / d_model)
        # Multi-Head Attention
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale
        # LayerNorm 1
        self.gamma1 = np.ones(d_model)
        self.beta1  = np.zeros(d_model)
        # FFN: deux couches lineaires avec activation GELU
        self.W1 = np.random.randn(d_ff, d_model) * scale
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_model, d_ff) * scale
        self.b2 = np.zeros(d_model)
        # LayerNorm 2
        self.gamma2 = np.ones(d_model)
        self.beta2  = np.zeros(d_model)

    def mha(self, x, mask=None):
        B, T, _ = x.shape
        Q = x @ self.W_Q.T
        K = x @ self.W_K.T
        V = x @ self.W_V.T
        # Split en tetes
        def split(t):
            return t.reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        Q, K, V = split(Q), split(K), split(V)
        # Attention scores
        scores = Q @ K.swapaxes(-2, -1) / np.sqrt(self.d_k)
        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)
        weights = softmax(scores)
        out = weights @ V
        # Concatener les tetes
        out = out.transpose(0, 2, 1, 3).reshape(B, T, self.d_model)
        return out @ self.W_O.T

    def ffn(self, x):
        h = gelu(x @ self.W1.T + self.b1)
        return h @ self.W2.T + self.b2

    def forward(self, x, mask=None):
        # Sous-couche 1: MHA + residual + LayerNorm (Pre-LN style)
        x = layer_norm(x + self.mha(x, mask), self.gamma1, self.beta1)
        # Sous-couche 2: FFN + residual + LayerNorm
        x = layer_norm(x + self.ffn(x), self.gamma2, self.beta2)
        return x

np.random.seed(42)
d_model, n_heads, d_ff = 64, 8, 256
batch, seq_len = 2, 10

block = TransformerBlock(d_model, n_heads, d_ff)
x = np.random.randn(batch, seq_len, d_model)
out = block.forward(x)

print("=== Transformer Block ===")
print(f"Input  : {x.shape}  (batch={batch}, seq={seq_len}, d_model={d_model})")
print(f"Output : {out.shape}  (meme shape = residual connections)")

# Nb de params
n_mha = 4 * d_model * d_model
n_ffn = 2 * d_model * d_ff
n_ln  = 2 * 2 * d_model
total = n_mha + n_ffn + n_ln
print(f"\nParametres par bloc:")
print(f"  MHA      : 4 * {d_model}^2 = {n_mha:,}")
print(f"  FFN      : 2 * {d_model} * {d_ff} = {n_ffn:,}")
print(f"  LayerNorm: {n_ln}")
print(f"  Total    : {total:,}")


---
## 2. Self-Attention en profondeur <a name='selfattention'></a>

### Types d'attention selon le masque
| Type | Masque | Modele | Usage |
|---|---|---|---|
| Bidirectionnel (full) | Aucun | BERT encoder | Comprehension |
| Causal (unidirectionnel) | Triangulaire inf. | GPT decoder | Generation |
| Cross-attention | Sur la source | Seq2Seq decoder | Traduction |
| Sparse attention | Pattern creux | Longformer, BigBird | Longues sequences |

### Complexite
- **Temps** : O(n^2 * d) ou n = longueur sequence, d = dimension
- **Memoire** : O(n^2) pour la matrice d'attention
- Goulot d'etranglement pour n > 2048 (Flash Attention resout cela)

### Flash Attention
- Reorganise les calculs pour eviter de materialiser la matrice n^2 en VRAM
- Calcule l'attention par blocs (tiling) dans la SRAM du GPU
- Meme resultat mathematique, 2-4x plus rapide, O(n) en memoire


In [ ]:
import numpy as np

# ============================================================
# TYPES D ATTENTION ET MASQUES
# ============================================================

def make_causal_mask(seq_len):
    """Masque triangulaire inferieur pour le decodeur autoregressive"""
    return np.tril(np.ones((seq_len, seq_len)))

def make_padding_mask(lengths, max_len):
    """Masque pour ignorer les tokens de padding"""
    batch = len(lengths)
    mask = np.zeros((batch, max_len))
    for i, l in enumerate(lengths):
        mask[i, :l] = 1
    return mask

def scaled_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    scores -= scores.max(axis=-1, keepdims=True)
    w = np.exp(scores)
    w = w / (w.sum(axis=-1, keepdims=True) + 1e-10)
    return w @ V, w

np.random.seed(42)
seq_len, d_k = 6, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

# 1. Full (bidirectionnel) -- BERT
_, w_full = scaled_attention(Q, K, V, mask=None)
print("=== 1. Attention bidirectionnelle (BERT) ===")
print("Matrice de poids (arrondie) :")
print(w_full.round(2))
print("=> Chaque token voit TOUS les autres (passe et futur)")

# 2. Causal -- GPT
causal = make_causal_mask(seq_len)
_, w_causal = scaled_attention(Q, K, V, mask=causal)
print("\n=== 2. Attention causale (GPT) ===")
print("Matrice de poids (arrondie) :")
print(w_causal.round(2))
print("=> Token t ne voit que t, t-1, t-2, ...")
print("   Triangulaire inferieur + 0 dans la partie superieure")


In [ ]:
import numpy as np

# ============================================================
# GROUPED QUERY ATTENTION (GQA) -- LLaMA 2, Mistral
# ============================================================
print("=== Grouped Query Attention (GQA) ===")
print("""
Probleme des MHA classiques:
  - Pour chaque couche: stocker K et V pour toutes les tetes => tres lourd en memoire (KV-cache)
  - Ex: LLaMA 65B avec seq=4096: KV-cache ~ plusieurs GB

Solutions:
  - MHA (Multi-Head Attention)     : n_kv_heads = n_heads       (original)
  - MQA (Multi-Query Attention)    : n_kv_heads = 1             (1 seule paire K/V partagee)
  - GQA (Grouped Query Attention)  : n_kv_heads = n_heads/G     (G groupes)

Exemples:
  - LLaMA 2 7B/13B  : MHA classique
  - LLaMA 2 70B     : GQA (n_heads=64, n_kv_heads=8)
  - Mistral 7B      : GQA (n_heads=32, n_kv_heads=8)
  - GPT-4           : probablement MQA ou GQA (non confirme)
""")

def grouped_query_attention(Q, K, V, n_heads, n_kv_heads, d_k):
    """
    Q : (batch, n_heads, seq_q, d_k)
    K : (batch, n_kv_heads, seq_k, d_k)
    V : (batch, n_kv_heads, seq_k, d_k)
    """
    B, _, T_q, _ = Q.shape
    T_k = K.shape[2]
    group_size = n_heads // n_kv_heads

    outputs = []
    for g in range(n_kv_heads):
        Q_group = Q[:, g*group_size:(g+1)*group_size, :, :]  # (B, group_size, T_q, d_k)
        K_g = K[:, g:g+1, :, :]   # (B, 1, T_k, d_k)
        V_g = V[:, g:g+1, :, :]   # (B, 1, T_k, d_k)
        # Broadcast K/V sur le groupe
        scores = Q_group @ K_g.swapaxes(-2,-1) / np.sqrt(d_k)
        scores -= scores.max(axis=-1, keepdims=True)
        w = np.exp(scores)
        w /= w.sum(axis=-1, keepdims=True) + 1e-10
        out = w @ V_g  # (B, group_size, T_q, d_k)
        outputs.append(out)

    return np.concatenate(outputs, axis=1)

np.random.seed(42)
B, n_heads, n_kv_heads = 2, 8, 2
seq_len, d_k = 10, 16
group_size = n_heads // n_kv_heads

Q = np.random.randn(B, n_heads, seq_len, d_k)
K = np.random.randn(B, n_kv_heads, seq_len, d_k)
V = np.random.randn(B, n_kv_heads, seq_len, d_k)

out = grouped_query_attention(Q, K, V, n_heads, n_kv_heads, d_k)
print(f"GQA output shape: {out.shape}")
print(f"n_heads={n_heads}, n_kv_heads={n_kv_heads}, group_size={group_size}")

# Calcul de la reduction du KV-cache
kv_mha = n_heads * 2 * d_k     # K et V pour chaque tete
kv_gqa = n_kv_heads * 2 * d_k  # K et V seulement pour n_kv_heads
print(f"\nKV-cache par token par couche:")
print(f"  MHA (n_kv={n_heads}): {kv_mha} floats")
print(f"  GQA (n_kv={n_kv_heads}): {kv_gqa} floats  (reduction: {kv_mha//kv_gqa}x)")


---
## 3. Encoder-Only : BERT et variantes <a name='encoder'></a>

### BERT (Bidirectional Encoder Representations from Transformers)
- **Architecture** : N blocs Transformer avec attention bidirectionnelle
- **Pre-entrainement** : MLM (15% masques) + NSP
- **Fine-tuning** : ajouter une tete de classification sur [CLS]

### Variantes
| Modele | Params | d_model | Heads | Layers | Difference |
|---|---|---|---|---|---|
| BERT-base | 110M | 768 | 12 | 12 | Original |
| BERT-large | 340M | 1024 | 16 | 24 | Plus grand |
| RoBERTa | 125M | 768 | 12 | 12 | Pas de NSP, plus de data |
| DistilBERT | 66M | 768 | 12 | 6 | Distillation de BERT |
| ALBERT | 12M | 128->768 | 12 | 12 | Cross-layer sharing |
| DeBERTa | 184M | 768 | 12 | 12 | Disentangled attention |

### Token [CLS]
- Token special ajoute en debut de sequence
- Son embedding en sortie represente la sequence entiere
- Utilise pour la classification

### Formule complete
```
Input: [CLS] w1 w2 ... wN [SEP]
Embedding = Token_emb + Position_emb + Segment_emb
x^l = LayerNorm(x^{l-1} + MHA(x^{l-1}))
x^l = LayerNorm(x^l + FFN(x^l))
Output[CLS] -> Linear -> Softmax (classification)
```


In [ ]:
import numpy as np

# ============================================================
# BERT ARCHITECTURE -- composants cles
# ============================================================

class BERTEmbedding:
    """
    BERT combine 3 types d embeddings:
    1. Token embedding (lookup table)
    2. Position embedding (appris, pas sinusoidal comme Transformer original)
    3. Segment embedding (phrase A vs phrase B pour NSP)
    """
    def __init__(self, vocab_size, max_pos, d_model, n_segments=2):
        self.token_emb    = np.random.randn(vocab_size, d_model) * 0.02
        self.position_emb = np.random.randn(max_pos,    d_model) * 0.02
        self.segment_emb  = np.random.randn(n_segments, d_model) * 0.02
        self.gamma = np.ones(d_model)
        self.beta  = np.zeros(d_model)

    def forward(self, token_ids, segment_ids=None):
        B, T = token_ids.shape
        if segment_ids is None:
            segment_ids = np.zeros_like(token_ids)

        tok_emb = self.token_emb[token_ids]
        pos_emb = self.position_emb[np.arange(T)]
        seg_emb = self.segment_emb[segment_ids]

        x = tok_emb + pos_emb + seg_emb
        # Layer Norm
        mu  = x.mean(axis=-1, keepdims=True)
        var = x.var(axis=-1, keepdims=True)
        x_norm = (x - mu) / np.sqrt(var + 1e-6)
        return self.gamma * x_norm + self.beta

# Simulation d un passage complet BERT
vocab_size, max_pos, d_model = 30522, 512, 768
bert_emb = BERTEmbedding(vocab_size, max_pos, d_model)

# Exemple: "[CLS] hello world [SEP] how are you [SEP]"
# [CLS]=101, [SEP]=102
token_ids   = np.array([[101, 7592, 2088, 102, 2129, 2024, 2017, 102]])
segment_ids = np.array([[0,   0,    0,    0,   1,    1,    1,    1  ]])  # phrase A vs B

x = bert_emb.forward(token_ids, segment_ids)
print("=== BERT Embedding ===")
print(f"Token IDs     : {token_ids[0]}")
print(f"Segment IDs   : {segment_ids[0]}")
print(f"Embedding shape: {x.shape}")
print(f"[CLS] embedding norm: {np.linalg.norm(x[0, 0]):.4f}")

# ============================================================
# BERT FINE-TUNING -- tetes de classification
# ============================================================
print("\n=== Tetes de classification BERT ===")

class BERTClassifier:
    """Classification de sequence via [CLS]"""
    def __init__(self, d_model, n_classes):
        self.W = np.random.randn(n_classes, d_model) * 0.02
        self.b = np.zeros(n_classes)

    def forward(self, cls_output):
        logits = cls_output @ self.W.T + self.b
        return logits

class BERTTokenClassifier:
    """Classification de tokens (NER, POS)"""
    def __init__(self, d_model, n_labels):
        self.W = np.random.randn(n_labels, d_model) * 0.02
        self.b = np.zeros(n_labels)

    def forward(self, sequence_output):
        return sequence_output @ self.W.T + self.b

# Exemples de taches
cls_output = x[0, 0]  # vecteur [CLS]
seq_output = x[0]     # tous les tokens

n_sentiment_classes = 3  # positif, neutre, negatif
n_ner_labels        = 9  # O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-MISC, I-MISC

clf_sent = BERTClassifier(d_model, n_sentiment_classes)
clf_ner  = BERTTokenClassifier(d_model, n_ner_labels)

logits_sent = clf_sent.forward(cls_output)
logits_ner  = clf_ner.forward(seq_output)

print(f"Sentiment (via [CLS])  : logits shape = {logits_sent.shape}")
print(f"NER (via tous tokens)  : logits shape = {logits_ner.shape}")

print("\n=== Caracteristiques cles de BERT ===")
specs = [
    ("Architecture",    "Encoder only, attention bidirectionnelle"),
    ("Pre-entrainement","MLM (15% masques) + NSP"),
    ("Position emb.",   "Apprises (pas sinusoidal) -- max 512 tokens"),
    ("Activation FFN",  "GELU"),
    ("Token special",   "[CLS] debut, [SEP] separation, [MASK] masque"),
    ("Fine-tuning",     "Ajouter une couche sur [CLS] ou sur tous les tokens"),
    ("Avantage",        "Excellent pour comprehension (NER, QA, classification)"),
    ("Limite",          "Pas adapte a la generation, max 512 tokens"),
]
for k, v in specs:
    print(f"  {k:20s}: {v}")


---
## 4. Decoder-Only : GPT, LLaMA, Mistral <a name='decoder'></a>

### Architecture GPT
- Uniquement des blocs decoder avec **masque causal**
- Pas de cross-attention (pas d'encoder)
- Generation **autoregressive** : genere un token a la fois
- Pre-entrainement : **Causal Language Modeling** (CLM)

### KV-Cache
- Au cours de la generation, on recalcule inutilement K et V pour tous les tokens precedents
- **KV-cache** : stocker K et V calcules, ne calculer que pour le nouveau token
- Reduit la complexite de O(n^2) a O(n) par nouveau token
- Cout : memoire proportionnelle a `n_layers * seq_len * n_heads * d_k * 2`

### Families de modeles
| Modele | Params | Context | Innovations |
|---|---|---|---|
| GPT-2 | 1.5B | 1024 | Causal LM sur WebText |
| GPT-3 | 175B | 2048 | In-context learning |
| LLaMA 1 | 7-65B | 2048 | Open source, efficace |
| LLaMA 2 | 7-70B | 4096 | GQA, RLHF, commercial |
| LLaMA 3 | 8-70B | 8192 | Meilleure qualite |
| Mistral 7B | 7B | 8192 | GQA + Sliding Window Attn |
| GPT-4 | ~1T? | 128k | MoE (speculation) |

### Innovations LLaMA vs GPT
| Feature | Transformer orig. | GPT-2 | LLaMA 2 |
|---|---|---|---|
| Normalisation | Post-LN | Post-LN | Pre-LN (RMSNorm) |
| Activation FFN | ReLU | GELU | SiLU/SwiGLU |
| Position encoding | Sinusoidal | Apprise | RoPE |
| Attention | MHA | MHA | GQA (70B) |


In [ ]:
import numpy as np

# ============================================================
# GPT DECODER BLOCK -- avec masque causal
# ============================================================

def softmax_stable(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=-1, keepdims=True) + 1e-10)

def rms_norm(x, weight, eps=1e-6):
    """RMSNorm -- plus simple que LayerNorm (pas de soustraction de la moyenne)
    Utilise dans LLaMA, Mistral, GPT-NeoX"""
    rms = np.sqrt((x**2).mean(axis=-1, keepdims=True) + eps)
    return weight * x / rms

def silu(x):
    """SiLU / Swish: x * sigmoid(x) -- utilise dans SwiGLU"""
    return x * (1 / (1 + np.exp(-np.clip(x, -500, 500))))

def swiglu(x, W1, W2, b1=None, b2=None):
    """
    SwiGLU: Swish(xW1) * xW2  -- activation FFN de LLaMA
    Remplace GELU + ameliore la performance
    """
    gate  = silu(x @ W1.T + (0 if b1 is None else b1))
    value = x @ W2.T      + (0 if b2 is None else b2)
    return gate * value

class GPTDecoderBlock:
    def __init__(self, d_model, n_heads, d_ff, use_swiglu=True):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads
        self.use_swiglu = use_swiglu

        s = np.sqrt(2.0 / d_model)
        # Attention
        self.W_Q = np.random.randn(d_model, d_model) * s
        self.W_K = np.random.randn(d_model, d_model) * s
        self.W_V = np.random.randn(d_model, d_model) * s
        self.W_O = np.random.randn(d_model, d_model) * s
        # RMSNorm
        self.rms1 = np.ones(d_model)
        self.rms2 = np.ones(d_model)
        # FFN (SwiGLU ou GELU)
        if use_swiglu:
            self.W_gate  = np.random.randn(d_ff, d_model) * s
            self.W_up    = np.random.randn(d_ff, d_model) * s
            self.W_down  = np.random.randn(d_model, d_ff) * s
        else:
            self.W1 = np.random.randn(d_ff, d_model) * s
            self.b1 = np.zeros(d_ff)
            self.W2 = np.random.randn(d_model, d_ff) * s
            self.b2 = np.zeros(d_model)

    def causal_attention(self, x):
        B, T, _ = x.shape
        Q = x @ self.W_Q.T
        K = x @ self.W_K.T
        V = x @ self.W_V.T

        def split(t):
            return t.reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
        Q, K, V = split(Q), split(K), split(V)

        scores = Q @ K.swapaxes(-2, -1) / np.sqrt(self.d_k)
        # Masque causal
        mask = np.tril(np.ones((T, T)))[np.newaxis, np.newaxis, :, :]
        scores = np.where(mask == 0, -1e9, scores)
        w = softmax_stable(scores)
        out = w @ V
        out = out.transpose(0, 2, 1, 3).reshape(B, T, self.d_model)
        return out @ self.W_O.T

    def forward(self, x):
        # Pre-LN + residual (style LLaMA)
        x = x + self.causal_attention(rms_norm(x, self.rms1))
        if self.use_swiglu:
            h = rms_norm(x, self.rms2)
            ffn_out = swiglu(h, self.W_gate, self.W_up) @ self.W_down.T
        else:
            h = rms_norm(x, self.rms2)
            ffn_out = np.maximum(0, h @ self.W1.T + self.b1) @ self.W2.T + self.b2
        x = x + ffn_out
        return x

np.random.seed(42)
d_model, n_heads, d_ff = 64, 8, 128
batch, seq_len = 2, 12

gpt_block = GPTDecoderBlock(d_model, n_heads, d_ff, use_swiglu=True)
x = np.random.randn(batch, seq_len, d_model)
out = gpt_block.forward(x)

print("=== GPT Decoder Block (avec SwiGLU + RMSNorm) ===")
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")

# Nombre de params
n_attn   = 4 * d_model * d_model
n_ffn_sg = 3 * d_model * d_ff   # SwiGLU a 3 matrices (gate, up, down)
n_ffn_gelu = 2 * d_model * d_ff + d_ff + d_model
n_norm   = 2 * d_model
print(f"\nParametres par bloc:")
print(f"  Attention (Q,K,V,O) : {n_attn:,}")
print(f"  FFN SwiGLU          : {n_ffn_sg:,}  (gate + up + down)")
print(f"  FFN GELU classique  : {n_ffn_gelu:,}")
print(f"  RMSNorm             : {n_norm}")
print(f"  Total SwiGLU        : {n_attn + n_ffn_sg + n_norm:,}")


In [ ]:
import numpy as np

# ============================================================
# KV-CACHE -- simulation
# ============================================================
print("=== KV-Cache : pourquoi et comment ===")
print("""
SANS KV-Cache (generation naive):
  Pour generer le token t, on recalcule K et V pour TOUS les tokens 1..t
  => O(t^2 * d) operaions au total pour generer T tokens
  => Tres inefficace!

AVEC KV-Cache:
  On stocke K[:t] et V[:t] en memoire
  Pour le token t, on calcule seulement K[t] et V[t], on les ajoute au cache
  => O(t * d) par nouveau token
  => Beaucoup plus rapide

Cout memoire du KV-Cache:
  n_layers * 2 * n_heads * d_k * seq_len * sizeof(float16)
""")

# Calcul du KV-Cache pour LLaMA 2
configs = [
    ("LLaMA 2 7B",  32, 32, 128, 4096),
    ("LLaMA 2 13B", 40, 40, 128, 4096),
    ("LLaMA 2 70B", 80, 8,  128, 4096),   # GQA: n_kv_heads=8
    ("Mistral 7B",  32, 8,  128, 8192),   # GQA: n_kv_heads=8
]
print(f"{'Modele':15s}  {'Layers':>6}  {'KV heads':>8}  {'d_k':>4}  {'Seq':>5}  {'KV-Cache (MB)':>14}")
for name, n_layers, n_kv, d_k, seq in configs:
    # float16 = 2 bytes
    kv_bytes = n_layers * 2 * n_kv * d_k * seq * 2
    kv_mb = kv_bytes / (1024**2)
    print(f"  {name:13s}  {n_layers:>6}  {n_kv:>8}  {d_k:>4}  {seq:>5}  {kv_mb:>12.1f} MB")

print("\n=> GQA reduit le KV-cache d un facteur n_heads/n_kv_heads")
print("   LLaMA 2 70B: 64/8 = 8x moins de KV-cache vs MHA standard!")

# ============================================================
# ROTARY POSITIONAL EMBEDDING (RoPE) -- LLaMA, Mistral
# ============================================================
print("\n=== RoPE (Rotary Position Embedding) ===")
def apply_rope(x, position):
    """
    Applique RoPE a un vecteur x a la position donnee
    Principe: rotate chaque paire (x_2i, x_{2i+1}) par l angle theta_i * position
    """
    d = x.shape[-1]
    half_d = d // 2
    thetas = 1.0 / (10000 ** (2 * np.arange(half_d) / d))
    angles = position * thetas

    cos_angles = np.cos(angles)
    sin_angles = np.sin(angles)

    x1, x2 = x[:half_d], x[half_d:]
    x_rotated = np.concatenate([
        x1 * cos_angles - x2 * sin_angles,
        x1 * sin_angles + x2 * cos_angles
    ])
    return x_rotated

q = np.random.randn(16)  # vecteur de query d_k=16
q_pos0 = apply_rope(q, position=0)
q_pos5 = apply_rope(q, position=5)

cos_sim = np.dot(q_pos0, q_pos5) / (np.linalg.norm(q_pos0) * np.linalg.norm(q_pos5))
print(f"  Avantages de RoPE vs PE absolu:")
print(f"  1. Encode la POSITION RELATIVE: dot(q_i, k_j) depend de (i-j)")
print(f"  2. Generalise a des sequences plus longues (avec scaling)")
print(f"  3. Compatible avec le KV-cache (applique directement sur Q et K)")
print(f"  Exemple: cos_sim(pos0, pos5) = {cos_sim:.4f}  (decroit avec la distance)")


---
## 5. Encoder-Decoder : T5, BART <a name='encdec'></a>

### Architecture
```
Source -> Encoder (attention bidirectionnelle) -> Memory
                                                     |
Target -> Decoder (masked self-attention) -> Cross-attention(Memory) -> Output
```

### Cross-Attention
- **Q** provient du decoder (ce que le decoder cherche)
- **K, V** proviennent de l'encoder (la source)
- Permet au decoder de "lire" la source a chaque pas de generation

### T5 (Text-to-Text Transfer Transformer)
- Toutes les taches formulees comme text-to-text
- `translate English to French: Hello` -> `Bonjour`
- `summarize: [article]` -> `[resume]`
- Objectif pre-entrainement : span denoising (remplace des spans par un seul masque)

### BART
- Encodeur: Transformer bidirectionnel (comme BERT)
- Decodeur: Transformer autoregressive (comme GPT)
- Pre-entrainement : reconstruire du texte bruisse (token masking, deletion, rotation...)
- Tres bon pour la summarisation et la traduction


In [ ]:
import numpy as np

# ============================================================
# CROSS-ATTENTION -- coeur de l architecture encoder-decoder
# ============================================================

def softmax_stable(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=-1, keepdims=True) + 1e-10)

class CrossAttention:
    """
    Q vient du decoder
    K, V viennent de l encoder (memory)
    => le decoder consulte la source a chaque pas
    """
    def __init__(self, d_model, n_heads):
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        s = np.sqrt(2.0 / d_model)
        self.W_Q = np.random.randn(d_model, d_model) * s
        self.W_K = np.random.randn(d_model, d_model) * s
        self.W_V = np.random.randn(d_model, d_model) * s
        self.W_O = np.random.randn(d_model, d_model) * s

    def forward(self, decoder_hidden, encoder_output, src_mask=None):
        """
        decoder_hidden  : (batch, tgt_len, d_model)  -- requetes
        encoder_output  : (batch, src_len, d_model)  -- cles et valeurs
        src_mask        : (batch, 1, 1, src_len)     -- ignorer le padding source
        """
        B, T_tgt, d_model = decoder_hidden.shape
        T_src = encoder_output.shape[1]

        Q = decoder_hidden  @ self.W_Q.T   # (B, T_tgt, d_model)
        K = encoder_output  @ self.W_K.T   # (B, T_src, d_model)
        V = encoder_output  @ self.W_V.T   # (B, T_src, d_model)

        def split(t, T):
            return t.reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)

        Q = split(Q, T_tgt)
        K = split(K, T_src)
        V = split(V, T_src)

        # Scores: (B, heads, T_tgt, T_src)
        scores = Q @ K.swapaxes(-2, -1) / np.sqrt(self.d_k)
        if src_mask is not None:
            scores = np.where(src_mask == 0, -1e9, scores)

        w = softmax_stable(scores)
        out = w @ V  # (B, heads, T_tgt, d_k)
        out = out.transpose(0, 2, 1, 3).reshape(B, T_tgt, d_model)
        return out @ self.W_O.T, w

np.random.seed(42)
d_model, n_heads = 64, 8
batch, src_len, tgt_len = 2, 8, 5

encoder_output  = np.random.randn(batch, src_len, d_model)
decoder_hidden  = np.random.randn(batch, tgt_len, d_model)

cross_attn = CrossAttention(d_model, n_heads)
output, attn_weights = cross_attn.forward(decoder_hidden, encoder_output)

print("=== Cross-Attention (Encoder-Decoder) ===")
print(f"Encoder output (source): {encoder_output.shape}  (batch, src_len={src_len}, d_model)")
print(f"Decoder hidden (target): {decoder_hidden.shape}  (batch, tgt_len={tgt_len}, d_model)")
print(f"Cross-attn output      : {output.shape}  (batch, tgt_len={tgt_len}, d_model)")
print(f"Attention weights      : {attn_weights.shape}  (batch, heads, tgt_len, src_len)")
print(f"\nChaque token cible 'lit' toute la source:")
print(f"  Poids tgt[0] sur src: {attn_weights[0, 0, 0].round(3)}")
print(f"  (sum={attn_weights[0, 0, 0].sum():.4f})")


In [ ]:
import numpy as np

# ============================================================
# T5 : span denoising vs BERT : token masking
# ============================================================
print("=== T5 Span Denoising vs BERT Token Masking ===")

def bert_masking_simple(tokens, mask_token="[MASK]", prob=0.15, seed=42):
    """BERT: masque 15% des tokens individuels"""
    rng = np.random.RandomState(seed)
    result = tokens[:]
    labels = {}
    for i, tok in enumerate(tokens):
        if rng.random() < prob:
            labels[i] = tok
            result[i] = mask_token
    return result, labels

def t5_span_denoising(tokens, noise_density=0.15, mean_span=3, seed=42):
    """T5: masque des spans entiers, remplace par <extra_id_X>"""
    rng = np.random.RandomState(seed)
    n = len(tokens)
    masked_input, targets = [], []
    i, sentinel_id = 0, 0
    while i < n:
        if rng.random() < noise_density:
            span_len = max(1, int(rng.poisson(mean_span)))
            span = tokens[i:i+span_len]
            masked_input.append(f"<extra_id_{sentinel_id}>")
            targets.append(f"<extra_id_{sentinel_id}>")
            targets.extend(span)
            sentinel_id += 1
            i += span_len
        else:
            masked_input.append(tokens[i])
            i += 1
    targets.append(f"<extra_id_{sentinel_id}>")
    return masked_input, targets

sentence = "The quick brown fox jumps over the lazy dog".split()
print(f"Original: {' '.join(sentence)}")

bert_in, bert_labels = bert_masking_simple(sentence)
print(f"\nBERT (token masking):")
print(f"  Input : {' '.join(bert_in)}")
print(f"  Labels: {bert_labels}")

t5_in, t5_out = t5_span_denoising(sentence, seed=7)
print(f"\nT5 (span denoising):")
print(f"  Input  : {' '.join(t5_in)}")
print(f"  Target : {' '.join(t5_out)}")

print("\n=== Quand utiliser quelle architecture ? ===")
comparison = [
    ("BERT (Encoder)",   "NER, classification, QA extractif",  "Pas de generation"),
    ("GPT (Decoder)",    "Generation, completion, chat",        "Pas optimal pour comprehension"),
    ("T5/BART (Enc-Dec)","Traduction, resume, QA generatif",   "Plus lourd, plus lent"),
]
for arch, strengths, limits in comparison:
    print(f"  {arch:20s}: strengths={strengths}")
    print(f"  {'':20s}  limits={limits}")


---
## 6. Positional Encodings <a name='pe'></a>

### Pourquoi ?
L'attention est **invariante a la permutation** : sans PE, le modele ne sait pas si un token est en position 1 ou 100.

### Types de PE

| Type | Modele | Formule | Avantage |
|---|---|---|---|
| Sinusoidal absolu | Transformer orig. | sin/cos(pos/10000^(2i/d)) | Generalise hors-distribution |
| Appris absolu | BERT, GPT-2 | Lookup table | Simple, flexible |
| Relatif (Shaw 2018) | T5, DeBERTa | Encode la distance relative | Generalise mieux |
| RoPE | LLaMA, Mistral | Rotation dans l espace complexe | Encode position relative, KV-cache friendly |
| ALiBi | MPT, BLOOM | Penalite lineaire sur les scores | Generalise a de longues sequences |


In [ ]:
import numpy as np

# ============================================================
# POSITIONAL ENCODINGS -- toutes les variantes
# ============================================================

def sinusoidal_pe(seq_len, d_model):
    """PE sinusoidal original (Vaswani 2017)"""
    PE = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, np.newaxis]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    PE[:, 0::2] = np.sin(pos * div)
    PE[:, 1::2] = np.cos(pos * div[:d_model//2 + d_model%2])[:, :d_model//2]
    return PE

def alibi_slopes(n_heads):
    """ALiBi: pentes pour chaque tete d attention"""
    m = np.arange(1, n_heads + 1)
    return 2 ** (-8 * m / n_heads)

def alibi_bias(seq_len, n_heads):
    """Matrice de biais ALiBi: penalite proportionnelle a la distance"""
    slopes = alibi_slopes(n_heads)
    positions = np.arange(seq_len)
    distances = np.abs(positions[:, np.newaxis] - positions[np.newaxis, :])
    # (n_heads, seq_len, seq_len)
    return -slopes[:, np.newaxis, np.newaxis] * distances[np.newaxis, :, :]

# 1. Sinusoidal PE
PE_sin = sinusoidal_pe(seq_len=20, d_model=32)
print("=== Positional Encoding Sinusoidal ===")
print(f"Shape: {PE_sin.shape}")
# Propriete: positions proches ont des PE similaires
sims = [np.dot(PE_sin[0], PE_sin[i]) / (np.linalg.norm(PE_sin[0]) * np.linalg.norm(PE_sin[i]))
        for i in range(8)]
print(f"Sim(pos0, pos_i) pour i=0..7: {[round(s,3) for s in sims]}")
print("=> Decroit avec la distance (inductive bias de localite)")

# 2. ALiBi
print("\n=== ALiBi (Attention with Linear Biases) ===")
n_heads, seq_len = 4, 6
slopes = alibi_slopes(n_heads)
print(f"Pentes par tete: {slopes.round(4)}")
bias = alibi_bias(seq_len, n_heads)
print(f"Biais ALiBi pour tete 0 (5 premiers tokens):")
print(bias[0, :5, :5].round(2))
print("=> Score_ij = Q_i.K_j^T / sqrt(d_k) - m_h * |i-j|")
print("   Pas de parametres supplementaires!")
print("   Generalise naturellement a des sequences plus longues")

# 3. RoPE -- propriete cle
print("\n=== RoPE : propriete de position relative ===")
print("dot(RoPE(q, pos_i), RoPE(k, pos_j)) depend uniquement de (pos_i - pos_j)")
print("=> L attention encode automatiquement la distance relative")
print("=> Fonctionne avec le KV-cache sans recalcul")
print("=> Peut etre etendu a des sequences plus longues via RoPE scaling")


---
## 7. Optimisations architecturales modernes <a name='optim'></a>

### Mixture of Experts (MoE)
- Au lieu d'une seule FFN, on a **E experts** (E FFN independantes)
- Un **routeur** (gating network) selectionne les top-k experts pour chaque token
- Params totaux = E * params_ffn, mais params actifs = k/E * params_totaux
- Modeles : Mixtral 8x7B, GPT-4 (speculation), Switch Transformer

### Flash Attention
- Probleme classique : la matrice d'attention n^2 ne rentre pas en SRAM
- Flash Attention : calcul par blocs (tiling), accumulation online du softmax
- Flash Attention 2 : meilleur parallelisme, 2x plus vite
- Flash Attention 3 : optimise pour H100 (FP8)

### Sliding Window Attention (Mistral)
- Chaque token ne voit que les W tokens precedents (fenetre glissante)
- Complexite O(n * W) au lieu de O(n^2)
- Les couches superieures ont un receptive field plus large

### Speculative Decoding
- Utiliser un petit modele (draft) pour proposer k tokens
- Valider avec le grand modele en un seul forward pass
- Si le grand modele est d'accord : k tokens generes au prix de 1
- Speedup typique : 2-3x


In [ ]:
import numpy as np

# ============================================================
# MIXTURE OF EXPERTS -- from scratch
# ============================================================
class MoELayer:
    """
    Sparse Mixture of Experts:
    - E experts (FFN independantes)
    - Routeur: choix des top-k experts par token
    """
    def __init__(self, d_model, d_ff, n_experts=8, top_k=2):
        self.d_model   = d_model
        self.n_experts = n_experts
        self.top_k     = top_k

        s = np.sqrt(2.0 / d_model)
        # E experts (chacun est une FFN)
        self.W1 = [np.random.randn(d_ff, d_model) * s for _ in range(n_experts)]
        self.W2 = [np.random.randn(d_model, d_ff) * s for _ in range(n_experts)]
        # Routeur: projette vers les logits d experts
        self.W_gate = np.random.randn(n_experts, d_model) * s

    def route(self, x):
        """Selectionne les top-k experts pour chaque token"""
        logits = x @ self.W_gate.T                    # (batch, seq, n_experts)
        # Softmax sur les experts
        e = np.exp(logits - logits.max(axis=-1, keepdims=True))
        probs = e / e.sum(axis=-1, keepdims=True)
        # Top-k experts
        top_k_idx   = np.argsort(probs, axis=-1)[..., -self.top_k:]
        top_k_probs = np.take_along_axis(probs, top_k_idx, axis=-1)
        # Re-normaliser les proba top-k
        top_k_probs = top_k_probs / top_k_probs.sum(axis=-1, keepdims=True)
        return top_k_idx, top_k_probs

    def forward(self, x):
        B, T, _ = x.shape
        top_k_idx, top_k_probs = self.route(x)
        output = np.zeros_like(x)

        for e in range(self.n_experts):
            # Trouver les tokens qui utilisent cet expert
            mask = (top_k_idx == e)   # (B, T, top_k)
            if not mask.any():
                continue
            # Appliquer l expert
            h = np.maximum(0, x @ self.W1[e].T)   # ReLU FFN
            expert_out = h @ self.W2[e].T

            # Ponderer par la proba du routeur
            for k in range(self.top_k):
                token_mask = mask[..., k]  # (B, T)
                w = top_k_probs[..., k][..., np.newaxis]   # (B, T, 1)
                output += token_mask[..., np.newaxis] * w * expert_out

        return output

np.random.seed(42)
d_model, d_ff = 32, 64
n_experts, top_k = 8, 2
batch, seq_len = 2, 6

moe = MoELayer(d_model, d_ff, n_experts, top_k)
x = np.random.randn(batch, seq_len, d_model)
out = moe.forward(x)

print("=== Mixture of Experts (MoE) ===")
print(f"Input:      {x.shape}")
print(f"Output:     {out.shape}")
print(f"n_experts:  {n_experts}, top_k={top_k}")
print(f"\nRoutage (batch 0, premiers tokens):")
idx, probs = moe.route(x)
for t in range(min(seq_len, 4)):
    experts_used = idx[0, t]
    weights      = probs[0, t]
    print(f"  Token {t}: experts={experts_used}, probs={weights.round(3)}")

# Stats d utilisation des experts
all_idx = idx.reshape(-1, top_k)
usage = np.bincount(all_idx.ravel(), minlength=n_experts)
print(f"\nUtilisation des experts: {usage}")
print(f"  Expert le plus utilise: {usage.argmax()} ({usage.max()} fois)")
print(f"  Load balancing ideal  : {len(all_idx)*top_k/n_experts:.1f} par expert")

# Calcul des params actifs vs totaux
params_total  = n_experts * (d_model * d_ff + d_ff * d_model)
params_active = top_k     * (d_model * d_ff + d_ff * d_model)
print(f"\nParams total  : {params_total:,}")
print(f"Params actifs : {params_active:,}  ({top_k}/{n_experts} = {top_k/n_experts:.0%} du total)")


---
## 8. Comparaison des architectures <a name='compare'></a>

### Recap : quel modele pour quelle tache ?

| Tache | Architecture | Exemple de modele |
|---|---|---|
| Classification de texte | Encoder-only | BERT, RoBERTa |
| NER | Encoder-only | BERT + token clf |
| QA extractif | Encoder-only | BERT + span extraction |
| Traduction | Encoder-Decoder | T5, NLLB, mBART |
| Summarisation | Encoder-Decoder | BART, T5, Pegasus |
| Generation libre | Decoder-only | GPT-4, LLaMA, Mistral |
| Chat / Instruction | Decoder-only + RLHF | ChatGPT, LLaMA-Chat |
| Code | Decoder-only | CodeLlama, StarCoder |
| Embeddings | Encoder-only | sentence-transformers, E5 |

### Taille des modeles courants (2024)
| Modele | Params | d_model | Layers | Heads | Context |
|---|---|---|---|---|---|
| BERT-base | 110M | 768 | 12 | 12 | 512 |
| BERT-large | 340M | 1024 | 24 | 16 | 512 |
| GPT-2 XL | 1.5B | 1600 | 48 | 25 | 1024 |
| LLaMA 2 7B | 7B | 4096 | 32 | 32 | 4096 |
| LLaMA 2 70B | 70B | 8192 | 80 | 64 | 4096 |
| Mistral 7B | 7B | 4096 | 32 | 32 | 8192 |
| Mixtral 8x7B | 47B* | 4096 | 32 | 32 | 32768 |

*47B params totaux mais ~13B actifs (MoE top-2/8)


In [ ]:
import numpy as np

# ============================================================
# CALCUL DU NOMBRE DE PARAMETRES D UN LLM
# ============================================================
def count_params_decoder(
    vocab_size, d_model, n_layers, n_heads, d_ff,
    n_kv_heads=None, use_swiglu=True
):
    """
    Compte le nombre de params d un decoder-only LLM
    """
    if n_kv_heads is None:
        n_kv_heads = n_heads

    # Embedding (token)
    emb = vocab_size * d_model

    # Par bloc decoder:
    # Attention: W_Q, W_K, W_V, W_O
    d_k = d_model // n_heads
    attn = (d_model * d_model +           # W_Q: (d_model, d_model)
            n_kv_heads * d_k * d_model +  # W_K: (n_kv * d_k, d_model)
            n_kv_heads * d_k * d_model +  # W_V: (n_kv * d_k, d_model)
            d_model * d_model)            # W_O: (d_model, d_model)

    # FFN
    if use_swiglu:
        ffn = 3 * d_model * d_ff   # gate, up, down
    else:
        ffn = 2 * d_model * d_ff   # W1, W2

    # RMSNorm (2 par bloc)
    norm = 2 * d_model

    per_block = attn + ffn + norm
    total_blocks = n_layers * per_block

    # LM head
    lm_head = vocab_size * d_model

    total = emb + total_blocks + lm_head
    return {
        "embedding": emb,
        "per_block": per_block,
        "total_blocks": total_blocks,
        "lm_head": lm_head,
        "total": total,
        "total_B": total / 1e9
    }

print("=== Calcul du nb de params ===")
models = [
    ("GPT-2 small",  50257, 768,  12, 12, 3072,  12,  False),
    ("LLaMA 2 7B",   32000, 4096, 32, 32, 11008, 32,  True),
    ("LLaMA 2 13B",  32000, 5120, 40, 40, 13824, 40,  True),
    ("LLaMA 2 70B",  32000, 8192, 80, 64, 28672, 8,   True),
    ("Mistral 7B",   32000, 4096, 32, 32, 14336, 8,   True),
]

print(f"  {'Modele':15s}  {'Emb(M)':>8}  {'Bloc(M)':>8}  {'Total(B)':>10}  {'Reel(B)':>8}")
official = {"GPT-2 small": 0.117, "LLaMA 2 7B": 7.0, "LLaMA 2 13B": 13.0,
            "LLaMA 2 70B": 70.0, "Mistral 7B": 7.2}
for name, vocab, d, n_lay, n_h, d_ff, n_kv, swiglu in models:
    p = count_params_decoder(vocab, d, n_lay, n_h, d_ff, n_kv, swiglu)
    ref = official.get(name, "?")
    print(f"  {name:15s}  {p['embedding']/1e6:>8.1f}  {p['per_block']/1e6:>8.1f}  {p['total_B']:>10.2f}  {str(ref):>8}")


---
## 9. Questions d’entretien <a name='questions'></a>

**Q : Quelle est la difference entre encoder-only, decoder-only et encoder-decoder ?**
> Encoder-only (BERT) : attention bidirectionnelle, voit tout le contexte => excellent pour comprendre. Decoder-only (GPT) : attention causale, genere un token a la fois => generation. Encoder-decoder (T5) : encoder lit la source, decoder genere => traduction, summarisation.

**Q : Pourquoi diviser par sqrt(d_k) dans l'attention ?**
> Quand d_k est grand, le produit scalaire Q.K^T a une grande variance (~ d_k). Diviser par sqrt(d_k) ramene la variance a 1, evitant la saturation du softmax et donc le vanishing gradient.

**Q : Qu'est-ce que le KV-cache et pourquoi est-il important ?**
> Pendant la generation, on stocke les K et V deja calcules pour eviter de les recalculer. Sans cache : O(n^2) operations pour generer n tokens. Avec cache : O(n). Goulot d'etranglement : la memoire (d'ou GQA/MQA).

**Q : RoPE vs PE sinusoidal vs PE appris ?**
> RoPE encode la position relative (dot(q_i, k_j) depend de i-j), generalise aux longues sequences, KV-cache friendly. PE sinusoidal : absolu, generalise hors-distribution. PE appris : simple, mais plafonne a la longueur max d'entrainement.

**Q : Qu'est-ce que la Mixture of Experts (MoE) ?**
> Au lieu d'une FFN dense, on a E FFN (experts) et un routeur qui selectionne top-k pour chaque token. Params actifs = k/E * params totaux => meme capacite de calcul pour plus de params totaux. Mixtral 8x7B : 47B params totaux, 13B actifs.

**Q : Pre-LN vs Post-LN : quelle est la difference ?**
> Post-LN (Transformer original) : LN apres la residuelle => entrainement instable pour les grands modeles. Pre-LN (GPT-2, LLaMA) : LN avant l'attention/FFN => plus stable, permet d'entrainer sans warmup agressif. GPT-3 et la plupart des LLMs modernes utilisent Pre-LN.

**Q : Quelle est la difference entre RMSNorm et LayerNorm ?**
> LayerNorm : soustrait la moyenne puis divise par l'ecart-type => 2 stats. RMSNorm : divise seulement par la RMS (racine de la moyenne des carres) => 1 stat, plus rapide, performances similaires. Utilise dans LLaMA, Mistral.

**Q : Expliquer Flash Attention en termes simples.**
> La matrice d'attention n^2 ne rentre pas en SRAM (memoire rapide du GPU). Flash Attention la calcule par blocs en utilisant le softmax online (algorithme de numeriquement stable) sans materialiser la matrice entiere. Meme resultat, O(n) memoire, 2-4x plus rapide.
